In [15]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain.agents import AgentExecutor, create_react_agent

load_dotenv()

# ==================== 定义工具 ====================
@tool
def add_tool(input_str: str) -> str:
    """
    计算两个数字的和。
    
    输入格式：数字1,数字2 或 数字1 数字2
    示例输入：3,5
    """
    try:
        # 解析输入，支持逗号或空格分隔
        parts = input_str.replace(',', ' ').split()
        if len(parts) != 2:
            return "错误：需要两个数字，用空格或逗号分隔。"
        a = float(parts[0])
        b = float(parts[1])
        result = a + b
        return f"{a} + {b} = {result}"
    except Exception as e:
        return f"计算错误：{e}"

@tool
def multiply_tool(input_str: str) -> str:
    """
    计算两个数字的乘积。
    
    输入格式：数字1,数字2 或 数字1 数字2
    """
    try:
        parts = input_str.replace(',', ' ').split()
        if len(parts) != 2:
            return "错误：需要两个数字，用空格或逗号分隔。"
        a = float(parts[0])
        b = float(parts[1])
        result = a * b
        return f"{a} × {b} = {result}"
    except Exception as e:
        return f"计算错误：{e}"

# 工具列表
tools = [add_tool, multiply_tool]

In [43]:
from langchain.memory import ConversationBufferWindowMemory

memory = ConversationBufferWindowMemory(
    memory_key="chat_history",  # 必须与模板中的变量名一致
    llm=llm,
    k=2,
    return_messages=True
)


# ==================== 配置 LLM ====================
llm = ChatOpenAI(
    model="deepseek-ai/DeepSeek-V3",
    openai_api_key=os.getenv("siliconFlow_API_KEY"),
    openai_api_base="https://api.siliconflow.cn/v1",
    temperature=0.3
)

# ==================== 创建提示词模板 ====================
# 这是 ReAct 代理的标准提示词模板
react_prompt = ChatPromptTemplate.from_template("""
你是一个乐于助人的助手。你可以使用以下工具：

{tools}

对话历史：
{chat_history}

请严格遵循以下格式：

Question: 用户的问题
Thought: 你应该思考要做什么
Action: 要采取的行动，必须是 [{tool_names}] 中的一个
Action Input: 行动的输入参数
Observation: 行动的结果
... (可以重复多次 Thought/Action/Action Input/Observation)
Thought: 我现在知道最终答案了
Final Answer: 对原始问题的最终答案

开始！

Question: {input}
Thought: {agent_scratchpad}
""")

# ==================== 创建 Agent ====================
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=react_prompt
)

# ==================== 创建 AgentExecutor ====================
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    memory=memory,
    verbose=True,               # 打印详细执行过程
    max_iterations=5,           # 最大工具调用次数，防止无限循环
    handle_parsing_errors=True  # 处理解析错误
)




C:\Users\naizz\AppData\Local\Temp\ipykernel_26512\939229699.py:3: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferWindowMemory(


In [53]:
result1 = agent_executor.invoke({"input": "请计算 3 + 5"})
print("最终答案：", result1["output"])



> Entering new AgentExecutor chain...
Thought: 我需要计算3加5的和
Action: add_tool
Action Input: 3,53.0 + 5.0 = 8.0我现在知道最终答案了
Final Answer: 3加5等于8

> Finished chain.
最终答案： 3加5等于8


In [54]:
print("测试2：复合问题（需要多步推理）")
print("=" * 50)
result2 = agent_executor.invoke({"input": "我有 12 个苹果，又买了 8 个，现在一共有多少个？"})
print("最终答案：", result2["output"])

测试2：复合问题（需要多步推理）


> Entering new AgentExecutor chain...
Thought: 用户想知道12个苹果加上8个苹果的总数，这是一个加法问题。
Action: add_tool
Action Input: 12,812.0 + 8.0 = 20.0我现在知道最终答案了
Final Answer: 你一共有20个苹果。

> Finished chain.
最终答案： 你一共有20个苹果。


In [55]:
result3 = agent_executor.invoke({"input": "我有多少个苹果"})
print(result3["output"])



> Entering new AgentExecutor chain...
Thought: 根据对话历史，用户之前提到有12个苹果，又买了8个，我已经计算过总共有20个苹果。
Final Answer: 你一共有20个苹果。

> Finished chain.
你一共有20个苹果。


In [52]:
result4 = agent_executor.invoke({"input": "我叫天下第一帅"})
print("最终答案：", result4["output"])



> Entering new AgentExecutor chain...
Thought: 用户只是在陈述自己的名字，不需要进行计算或工具操作。
Final Answer: 好的，我知道了你的名字是"天下第一帅"。

> Finished chain.
最终答案： 好的，我知道了你的名字是"天下第一帅"。


In [56]:
result5 = agent_executor.invoke({"input": "刚才我说我叫什么？然后帮我计算 7 + 8"})
print("回答:", result5["output"])



> Entering new AgentExecutor chain...
Thought: 首先需要确认用户之前是否提到过自己的名字，然后计算7加8的和。
Action: add_tool
Action Input: 7,87.0 + 8.0 = 15.0Thought: 用户之前没有提到过自己的名字，无法回答名字的问题。现在可以回答7加8的结果。
Final Answer: 7加8等于15。关于你的名字，之前的对话中没有提到过。

> Finished chain.
回答: 7加8等于15。关于你的名字，之前的对话中没有提到过。


In [34]:
# 查看记忆中的消息
print("当前记忆内容：")
for msg in memory.chat_memory.messages:
    print(f"{msg.type}: {msg.content}")

当前记忆内容：
human: 请计算 3 + 5
ai: 3 + 5 = 8
human: 我有 12 个苹果，又买了 8 个，现在一共有多少个？
ai: 你有12个苹果，又买了8个，现在一共有20个苹果。
human: 我有多少个苹果
ai: 你现在一共有20个苹果。
human: 计算 7 乘以 8
ai: 7 乘以 8 等于 56
human: 计算 7 乘以 8
ai: 7 乘以 8 等于 56
human: 刚才我说我叫什么？然后帮我计算 7 + 8
ai: 对话历史中没有提到您的名字。7 + 8 = 15
human: 我叫天下第一帅
ai: 好的，我记住了您的名字是“天下第一帅”。
human: 刚才我说我叫什么？然后帮我计算 7 + 8
ai: 您之前告诉我您的名字是“天下第一帅”。7 + 8 = 15


In [41]:
# 查看记忆中的消息
print("当前记忆内容：")
for msg in memory.chat_memory.messages:
    print(f"{msg.type}: {msg.content}")

当前记忆内容：
human: 请计算 3 + 5
ai: 3 + 5 = 8
human: 我有 12 个苹果，又买了 8 个，现在一共有多少个？
ai: 你有12个苹果，又买了8个，现在一共有20个苹果。
human: 我有多少个苹果
ai: 你现在一共有20个苹果。
human: 我叫天下第一帅
ai: 你好，天下第一帅！请问有什么我可以帮助你的吗？
human: 刚才我说我叫什么？然后帮我计算 7 + 8
ai: 你刚才说你叫“天下第一帅”，7 + 8 = 15


In [51]:
# 查看记忆中的消息
print("当前记忆内容：")
for msg in memory.chat_memory.messages:
    print(f"{msg.type}: {msg.content}")

当前记忆内容：
human: 我有 12 个苹果，又买了 8 个，现在一共有多少个？
ai: 你现在一共有20个苹果。
human: 请计算 3 + 5
ai: 3 + 5 = 8
human: 我叫天下第一帅
ai: 你好，天下第一帅！有什么我可以帮助你的吗？
human: 我有多少个苹果
ai: 我无法确定你有多少个苹果，因为你没有提供相关的数量信息。
human: 刚才我说我叫什么？然后帮我计算 7 + 8
ai: 你之前说你叫"天下第一帅"，另外7加8等于15
human: 请计算 3 + 5
ai: 3加5等于8
human: 刚才我说我叫什么？然后帮我计算 7 + 8
ai: 你之前说你叫"天下第一帅"，另外7加8等于15


In [57]:
# 查看记忆中的消息
print("当前记忆内容：")
for msg in memory.chat_memory.messages:
    print(f"{msg.type}: {msg.content}")

当前记忆内容：
human: 我有 12 个苹果，又买了 8 个，现在一共有多少个？
ai: 你现在一共有20个苹果。
human: 请计算 3 + 5
ai: 3 + 5 = 8
human: 我叫天下第一帅
ai: 你好，天下第一帅！有什么我可以帮助你的吗？
human: 我有多少个苹果
ai: 我无法确定你有多少个苹果，因为你没有提供相关的数量信息。
human: 刚才我说我叫什么？然后帮我计算 7 + 8
ai: 你之前说你叫"天下第一帅"，另外7加8等于15
human: 请计算 3 + 5
ai: 3加5等于8
human: 刚才我说我叫什么？然后帮我计算 7 + 8
ai: 你之前说你叫"天下第一帅"，另外7加8等于15
human: 我叫天下第一帅
ai: 好的，我知道了你的名字是"天下第一帅"。
human: 请计算 3 + 5
ai: 3加5等于8
human: 我有 12 个苹果，又买了 8 个，现在一共有多少个？
ai: 你一共有20个苹果。
human: 我有多少个苹果
ai: 你一共有20个苹果。
human: 刚才我说我叫什么？然后帮我计算 7 + 8
ai: 7加8等于15。关于你的名字，之前的对话中没有提到过。


In [58]:
!git add .


In [60]:
!git commit -m"Today, I implemented multi-turn conversations for the agent using ReAct."


On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   Untitled.ipynb

no changes added to commit (use "git add" and/or "git commit -a")
